In [ ]:
''''
import pandas as pd
import geopandas as gpd
import numpy as np
from skimage.filters import threshold_otsu
import rasterio
from rasterio.merge import merge
from rasterio.features import shapes
import os
from shapely.geometry import box
'''
def create_census_geojson(input_csv, input_shp, output_dir="data/"):
    """
    Creates a census GeoJSON file from CSV and shapefile data.
    
    Parameters:
        input_csv (str): Path to the census CSV file.
        input_shp (str): Path to the census shapefile.
        output_dir (str): Directory where the censustracts.geojson will be saved. 
                         Default is "data/". For city-specific, use "data/cityname/".
    """
    import pandas as pd
    import geopandas as gpd
    import os
    
    canada_csv = input_csv
    shapefile_path = input_shp

    df = pd.read_csv(canada_csv, encoding='latin1')
    df = df.apply(pd.to_numeric, errors='coerce')
    print(df.head())
    # Drop columns from a pandas DataFrame
    df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd'])  
    #print(df.head())

    df = pd.read_csv(canada_csv, encoding='latin1')
    # Drop columns from a pandas DataFrame
    df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd']) 
    print(df.head())

    gdf = gpd.read_file(shapefile_path)
    gdf = gdf.to_crs(epsg=4326) # Ensure both are in the same CRS

    # merging the csv data with the shapefile data
    merged = gdf.merge(df, left_on='CTUID', right_on='CTUID')
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Export to GeoJSON
    geojson_path = os.path.join(output_dir, "censustracts.geojson")
    merged.to_file(geojson_path, driver="GeoJSON")
    print(f"Census tracts GeoJSON saved to: {geojson_path}")
    return geojson_path

    #example usage
    #create_census_geojson("raw-data/montreal/9810001402-eng_clean.csv","raw-data/census tracts/lct_000b21a_e.shp", "data/montreal/")


jupyter nbconvert --to script your_notebook.ipynb in order to convert this markdown file to python script for calls

In [ ]:
'''# Path to your shapefile
shapefile_path = "raw-data/census tracts/lct_000b21a_e.shp"

# Read the shapefile using geopandas
gdf = gpd.read_file(shapefile_path)

#convert to WGS84
gdf = gdf.to_crs(epsg=4326) 

# Export to GeoJSON
geojson_path = "raw-data/montreal/lct_000b21a_e.geojson"
gdf.to_file(geojson_path, driver="GeoJSON")
'''
# --- IGNORE --- we dont really need to run this part, just for reference

In [ ]:
'''
#this block will create and export a new geojson with the csv data merged in
# Read CSV and shapefile
csv_path = "raw-data/montreal/9810001402-eng_clean.csv"
shapefile_path = "raw-data/census tracts/lct_000b21a_e.shp"

df = pd.read_csv(csv_path, encoding='latin1')
# Drop columns from a pandas DataFrame
df = df.drop(columns=['pop16', 'pop%delt', 'dwelltot', 'privd']) 
print(df.head())

gdf = gpd.read_file(shapefile_path)
gdf = gdf.to_crs(epsg=4326) # Ensure both are in the same CRS

# merging the csv data with the shapefile data
merged = gdf.merge(df, left_on='CTUID', right_on='CTUID')
# Export to GeoJSON as bound_
geojson_path = os.path.join("data/", "censustracts.geojson")
merged.to_file(geojson_path, driver="GeoJSON")'''

so this section is dedicated to exclusively raster operators 
i want to show and ndvi (NIR-RED/NIR+RED wavelength) index with a threshold that will only display 'healthy vegetation' the datasets for this have been picked during peak growth cycles because winter obviously doesnt have much vegetation
landsat 2 collection 2 is atmospherically compensated but needs a scaling corrective factor of (band*0.0000275-0.2) 
i will used bands 4 for red and 5 for near infra-red

In [ ]:
#scaling function for landsat 2 collection 2
# Path to your input .tiff file
#input_tiff = "data/landsat/your_image.tiff" # this is just an example path
#output_tiff = "data/landsat/scaled_image.tiff" #the output path

def scale_tiff(input_tiff, prefix="scaled_"):
    
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
    
    # Get directory and filename
    dir_name, file_name = os.path.split(input_tiff)
    output_tiff = os.path.join(dir_name, prefix + file_name)

    with rasterio.open(input_tiff) as src:
        profile = src.profile
        data = src.read(1)  # Read the first band

        # Apply scaling: (DN * 0.0000275) - 0.2
        scaled_data = (data * 0.0000275) - 0.2

        # Update profile to float32 for scaled values
        profile.update(dtype=rasterio.float32)

        # Write the scaled data to a new .tiff file
        with rasterio.open(output_tiff, 'w', **profile) as dst:
            dst.write(scaled_data.astype(rasterio.float32), 1)

    print("Scaling complete. Output saved to:", output_tiff)
    return output_tiff

# montreal usage:

#scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF")
#band 4
#scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF")
#band 5


In [ ]:
# make ndvi function


def calculate_ndvi(red_band_path, nir_band_path, city):
    import numpy as np
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
   # Get directory and filename
    dir_name, file_name = os.path.split(red_band_path)
    output_tiff = os.path.join(dir_name, "ndvi_" + city)
    # Open red band
    with rasterio.open(red_band_path) as red_src:
        red = red_src.read(1).astype('float32')
        profile = red_src.profile

    # Open NIR band
    with rasterio.open(nir_band_path) as nir_src:
        nir = nir_src.read(1).astype('float32')

    # NDVI calculation: (NIR - RED) / (NIR + RED)
    ndvi = (nir - red) / (nir + red)
    ndvi = np.clip(ndvi, -1, 1)  # Optional: clip values to valid NDVI range

    # Update profile for output
    profile.update(dtype=rasterio.float32, count=1)

    # Write NDVI to new tiff
    with rasterio.open(output_tiff, 'w', **profile) as dst:
        dst.write(ndvi, 1)

    print("NDVI calculation complete. Output saved to:", output_tiff)

# Example usage for montreal:
#calculate_ndvi("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF", "raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF",city="montreal.tiff")

we have an ndvi map now. we need to find a threshold for it. 
i have found that the otsu method might be best for python automation

In [ ]:
def otsu_ndvi_threshold(ndvi_tiff):
    import numpy as np
    from skimage.filters import threshold_otsu
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
    # Read NDVI raster
    with rasterio.open(ndvi_tiff) as src:
        ndvi = src.read(1)
        profile = src.profile

    # Flatten and remove NaN values for threshold calculation
    ndvi_flat = ndvi.flatten()
    ndvi_flat = ndvi_flat[~np.isnan(ndvi_flat)]

    # Compute Otsu threshold
    thresh = threshold_otsu(ndvi_flat)
    print(f"Otsu threshold for NDVI: {thresh}")

    # Create binary mask: 1 for healthy vegetation, 0 otherwise
    mask = (ndvi >= thresh).astype(np.uint8)

    # Save mask as new tiff
    profile.update(dtype=rasterio.uint8, count=1)
    mask_tiff = ndvi_tiff.replace('.tiff', '_mask.tiff')
    with rasterio.open(mask_tiff, 'w', **profile) as dst:
        dst.write(mask, 1)

    print("Mask saved to:", mask_tiff)
    return thresh, mask_tiff

# Example usage montreal:
#otsu_ndvi_threshold("raw-data/montreal/landsat2_c2/ndvi_montreal.tiff")

In [ ]:
# turn the mask into a geojson since it will show the healthy vegetation areas based on the threhold value
def mask_to_geojson(mask_tiff):
    import pandas as pd
    import geopandas as gpd
    import numpy as np
    from skimage.filters import threshold_otsu
    import rasterio
    from rasterio.merge import merge
    from rasterio.features import shapes
    import os
    from shapely.geometry import box
    
    # Get directory and filename
    dir_name, file_name = os.path.split(mask_tiff)
    # Get parent directory (one level above)
    parent_dir = os.path.dirname(dir_name)
    # Remove ".tiff" or ".tif" from filename
    base_name = file_name.replace('.tiff', '').replace('.tif', '')
    base_name = base_name.replace('_mask', '')  # Remove '_mask' from filename
    output_geojson = os.path.join(parent_dir, f"greenspace_{base_name}.geojson")

    with rasterio.open(mask_tiff) as src:
        mask = src.read(1)
        mask = mask.astype('uint8')
        transform = src.transform
        crs = src.crs

        # Extract shapes (polygons) where mask == 1
        results = (
            {"properties": {"value": v}, "geometry": s}
            for s, v in shapes(mask, mask=mask==1, transform=transform)
            if v == 1
        )

        gdf = gpd.GeoDataFrame.from_features(list(results))
        gdf = gdf.set_crs(crs)
        gdf = gdf.to_crs(epsg=4326)  # Ensure both are in the same CRS

        # Save to parent directory
        gdf.to_file(output_geojson, driver="GeoJSON")
        print(f"GeoJSON saved to: {output_geojson}")
        return output_geojson

# Example usage montreal:

#mask_to_geojson("raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff")

in theory now, for other cities i can just call the functions and avoid a lot of code repetitions. 

In [ ]:
# code to cut geographic extent of a geojson to a specific bounding box know that the bounds should be the same as the map bounds in webapp.js
#since we are pre processing and cities are different sizes, this needs to be  manually determined and entered for each geojson city

# Load greenspace polygons
#greenspace_gdf = gpd.read_file("raw-data/montreal/landsat2_c2/output_simplified.geojson")

# Define your bounding box (minx, miny, maxx, maxy)
#minx, miny = -74.00, 45.35
#maxx, maxy = -73.40, 45.75
#bbox = box(minx, miny, maxx, maxy)

# Clip greenspace polygons to bounding box
#clipped_gdf = greenspace_gdf.clip(bbox)

# Save clipped GeoJSON
#clipped_gdf.to_file("data/montreal/output_simplified_clipped.geojson", driver="GeoJSON")


def clip_geojson_to_bbox(input_file, output_dir, bbox_coords=None):
    import geopandas as gpd
    from shapely.geometry import box
    import os
    input_file = os.path.abspath(input_file)
    input_dir = os.path.dirname(input_file)
    if bbox_coords is None:
        # Manual input mode
        print("Enter the bounding box coordinates:")
        minx = float(input("Minimum longitude (minx): "))
        miny = float(input("Minimum latitude (miny): "))
        maxx = float(input("Maximum longitude (maxx): "))
        maxy = float(input("Maximum latitude (maxy): "))
    else:
        # Automated mode - expect a tuple/list: (minx, miny, maxx, maxy)
        minx, miny, maxx, maxy = bbox_coords
        print(f"Using bounding box: ({minx}, {miny}, {maxx}, {maxy})")
    try:
        greenspace_gdf = gpd.read_file(input_file)
        greenspace_gdf = greenspace_gdf.to_crs(epsg=4326)
    except Exception as e:
        print(f"Error reading the file: {e}")
        return
    bbox = box(minx, miny, maxx, maxy)
    try:
        clipped_gdf = greenspace_gdf.clip(bbox)
    except Exception as e:
        print(f"Error clipping the GeoJSON: {e}")
        return
    try:
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, "greenspace_only.geojson")
        clipped_gdf.to_file(output_file, driver="GeoJSON")
        print(f"Clipped GeoJSON saved to output directory: {output_file}")
        
        # Also save to input directory as backup
        input_output_file = os.path.join(input_dir, "greenspace_only.geojson")
        clipped_gdf.to_file(input_output_file, driver="GeoJSON")
        print(f"Clipped GeoJSON also saved to input directory: {input_output_file}")
    except Exception as e:
        print(f"Error saving the clipped GeoJSON: {e}")


# Call example the function
#clip_geojson_to_bbox("raw-data/montreal/output_simplified.geojson", output_dir="data/montreal")


In [ ]:
#should be a deprecated block now since we made the function above
#  ----------- for this to work, we must find how much greenspace area is within each census tract
# the we create a column where we divide the population by the greenspace area to get a per capita value
# later we can use all that data to generate a rating for how green the city is
# Load greenspace polygons (GeoJSON)
#
#
#
# im hoping to skip this by forcing census tracts to display on top. reduces data size this way
# we still should make this file for the statistical math to be easier

#greenspace_gdf = gpd.read_file("data/montreal/output_simplified_clipped.geojson")

# Load census tracts (GeoJSON or Shapefile)
#census_gdf = gpd.read_file("data/censustracts.geojson")

# Ensure both are in the same CRS
#greenspace_gdf = greenspace_gdf.to_crs(epsg=4326)
#census_gdf = census_gdf.to_crs(epsg=4326)

# Spatial join: append census tract attributes to greenspace polygons
#joined = gpd.sjoin(greenspace_gdf, census_gdf, how="left", predicate="intersects")

# Save the result
#joined.to_file("raw-data/montreal/greenspace_with_census.geojson", driver="GeoJSON")

#print("Processing complete. file  saved to data/montreal/greenspace_with_census.geojson")



# im hoping to skip this by forcing census tracts to display on top. reduces data size this way
    # we still should make this file for the statistical math to be easier
# Save the clipped GeoJSON in the same directory as the input file
def combine_greenspace_with_census(input_file, output_dir, census_dir=None):
    import geopandas as gpd
    import os

    # Convert to absolute paths to avoid any confusion
    input_file = os.path.abspath(input_file)
    output_dir = os.path.abspath(output_dir)
    
    try:
        # Load the clipped GeoJSON file
        clipped_gdf = gpd.read_file(input_file)
        print(f"Clipped GeoJSON loaded from: {input_file}")
    except Exception as e:
        print(f"Error reading the clipped GeoJSON file: {e}")
        return None

    try:
        # Load the census GeoJSON file from the city-specific directory
        # If census_dir is not provided, try to find it in the parent directory of output_dir
        if census_dir is None:
            # Assume the census file is in the same directory structure
            base_dir = os.path.dirname(os.path.dirname(output_dir))  # Get base directory
            # Extract city name from output_dir
            city_name = os.path.basename(output_dir)
            census_path = os.path.join(base_dir, "data", city_name, "censustracts.geojson")
        else:
            census_path = os.path.join(census_dir, "censustracts.geojson")
            
        census_gdf = gpd.read_file(census_path)
        print(f"Census GeoJSON loaded from: {census_path}")
    except Exception as e:
        print(f"Error reading the censustracts file from {census_path}: {e}")
        return None

    # Ensure both GeoDataFrames are in the same CRS
    clipped_gdf = clipped_gdf.to_crs(epsg=4326)
    census_gdf = census_gdf.to_crs(epsg=4326)

    try:
        # Spatial join: append census tract attributes to greenspace polygons
        joined = gpd.sjoin(clipped_gdf, census_gdf, how="left", predicate="intersects")
    except Exception as e:
        print(f"Error performing spatial join: {e}")
        return None

    # Save the combined GeoJSON in the specified output directory
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    combined_file = os.path.join(output_dir, "greenspace_with_census.geojson")
    try:
        joined.to_file(combined_file, driver="GeoJSON")
        print(f"Combined GeoJSON saved to: {combined_file}")
        return combined_file
    except Exception as e:
        print(f"Error saving the combined GeoJSON: {e}")
        return None

#combine_greenspace_with_census("data/montreal/greenspace_only.geojson", output_dir="raw-data/montreal")



we will now create a pipeline to produce a statistical representation of the census tracts
this will also be used to create summary stats
up till here, all good

In [ ]:
'''
import geopandas as gpd
import pandas as pd
import os
# Load census tracts (GeoJSON or Shapefile)
greenspace_capita = gpd.read_file("raw-data/montreal/output_simplified_clipped_with_census.geojson")
greenspace_capita = greenspace_capita.to_crs(epsg=32188) # Ensure  CRS

# Calculate greenspace area per feature (in square meters)
greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
greenspace_capita['LANDAREA']= greenspace_capita['LANDAREA']*1e6

# Ensure 'CTUID' and 'pop21' are valid
greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

# Consolidate greenspace area by CTUID (sum all polygons in each tract)
greenspace_sum = greenspace_capita.groupby('CTUID').agg({
    'greenspace_area': 'sum',
    'pop21': 'first',
    'LANDAREA': 'first',
    'geometry': 'first'
}).reset_index()

# Calculate greenspace per tract (area/land area)m
greenspace_sum['greenspace_per_tract'] = greenspace_sum['greenspace_area'] / greenspace_sum['LANDAREA']

# Calculate greenspace per capita (area/population) m/person
greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']

# Save consolidated GeoJSON
greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
greenspace_sum = greenspace_sum.to_crs(epsg=4326)
greenspace_sum.to_file("raw-data/montreal/greenspace_capita.geojson", driver="GeoJSON")
# do math, cut the polygons, check the percentage of greenspace per census tract,use that percentage to get area of greenspace, 
# then divide population by greenspace area to get per capita value, then use that to generate a rating for how green the city is
# the file above should already have the census data merged and the greenspace polygons cut to the census tracts
#remove the census tract data from the greenspace polygons to reduce file size
greenspace_final = gpd.read_file("raw-data/montreal/greenspace_capita.geojson")
greenspace_final = greenspace_final.drop(columns=['CTNAME', 'PRUID', 'CDUID', 'CDNAME', 'CSDUID', 'CSDNAME', 'CMAUID', 'CMANAME', 'CDTYPE', 'CSDTYPE', 'TYPE', 'pop21', 'LANDAREA'], errors='ignore')
greenspace_final.to_file("data/montreal/greenspace_final.geojson", driver="GeoJSON")

'''

def process_greenspace_census_data(input_file, output_dir):
    import geopandas as gpd
    import pandas as pd
    import os
    greenspace_capita = gpd.read_file(input_file)
    greenspace_capita = greenspace_capita.to_crs(epsg=32188) # Ensure  CRS
    input_dir = os.path.dirname(input_file)


    # Calculate greenspace area per feature (in square meters)
    greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
    greenspace_capita['LANDAREA']= greenspace_capita['LANDAREA']*1e6

    # Ensure 'CTUID' and 'pop21' are valid
    greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
    greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
    greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

    # Consolidate greenspace area by CTUID (sum all polygons in each tract)
    greenspace_sum = greenspace_capita.groupby('CTUID').agg({
        'greenspace_area': 'sum',
        'pop21': 'first',
        'LANDAREA': 'first',
        'geometry': 'first'
    }).reset_index()

    # Calculate greenspace per tract (area/land area)m
    greenspace_sum['greenspace_per_tract'] = greenspace_sum['greenspace_area'] / greenspace_sum['LANDAREA']

    # Calculate greenspace per capita (area/population) m/person
    greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']

    # Save consolidated GeoJSON
    save_dir = os.path.join(input_dir, os.path.basename(input_file).replace("greenspace_with_census.geojson", "greenspace_capita.geojson"))
    greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
    greenspace_sum = greenspace_sum.to_crs(epsg=4326)
    greenspace_sum.to_file(save_dir, driver="GeoJSON")
    print(f"Combined GeoJSON saved to: {save_dir}")

# Example usage
# process_greenspace_census_data("raw-data/montreal/greenspace_with_census.geojson", output_dir="data/montreal")




In [ ]:
#we use this function now
def process_greenspace_data(input_file, output_dir):
    """
    Processes greenspace data to calculate greenspace per tract and per capita.

    Parameters:
        input_file (str): Path to the input GeoJSON file containing greenspace and census data.
        output_dir (str): Directory where the output files will be saved. Default is "data".

    Outputs:
        - greenspace_capita.geojson: GeoJSON with greenspace per tract and per capita values.
        - greenspace_final.geojson: GeoJSON with reduced columns for final use.
    """
    import geopandas as gpd
    import pandas as pd
    import os
    import numpy as np
    #print(f"we are at processing greenspace data below")
    try:
        # Load greenspace census data
        input_dir = os.path.dirname(input_file)
        save_dir = os.path.join(input_dir, os.path.basename(input_file))
        greenspace_capita = gpd.read_file(input_file)
        greenspace_capita = greenspace_capita.to_crs(epsg=32188)  # Ensure CRS is in meters for area calculations

        # Calculate greenspace area per feature (in square meters)
        greenspace_capita['greenspace_area'] = greenspace_capita.geometry.area
        greenspace_capita['LANDAREA'] = greenspace_capita['LANDAREA'] * 1e6  # Convert LANDAREA to square meters

        # Ensure 'CTUID' and 'pop21' are valid
        greenspace_capita['CTUID'] = greenspace_capita['CTUID'].astype(str)
        greenspace_capita['pop21'] = pd.to_numeric(greenspace_capita['pop21'], errors='coerce')
        greenspace_capita['LANDAREA'] = pd.to_numeric(greenspace_capita['LANDAREA'], errors='coerce')

        # Consolidate greenspace area by CTUID (sum all polygons in each tract)
        greenspace_sum = greenspace_capita.groupby('CTUID').agg({
            'greenspace_area': 'sum',
            'pop21': 'first',
            'LANDAREA': 'first',
            'geometry': 'first'
        }).reset_index()

        # Calculate greenspace per capita (area/population) in m²/person
        greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']
       
        # Compute per-capita (area / population) safely
        greenspace_sum['greenspace_per_capita'] = greenspace_sum['greenspace_area'] / greenspace_sum['pop21']
        # Areas with no greenspace -> 0
        greenspace_sum.loc[greenspace_sum['greenspace_area'] == 0, 'greenspace_per_capita'] = 0.0
        # Areas with no population (pop21 is NaN or <= 0) -> -1
        no_pop_mask = greenspace_sum['pop21'].isna() | (greenspace_sum['pop21'] <= 0)
        greenspace_sum.loc[no_pop_mask, 'greenspace_per_capita'] = -1.0

        # Save consolidated GeoJSON
        save_dir = os.path.join(input_dir, os.path.basename(input_file))
        greenspace_sum = gpd.GeoDataFrame(greenspace_sum, geometry='geometry', crs='EPSG:32188')
        greenspace_sum = greenspace_sum.to_crs(epsg=4326)  # Convert back to WGS84 for GeoJSON
        greenspace_sum.to_file(save_dir, driver="GeoJSON")
        print(f"Greenspace capita data saved to: {save_dir}")

        greenspace_sum = greenspace_sum.to_crs(epsg=32188)  # Convert back to projected CRS for area calculations

        # Remove unnecessary columns to reduce file size
        greenspace_final = greenspace_sum.drop(columns=[
            'CTNAME', 'PRUID', 'CDUID', 'CDNAME', 'CSDUID', 'CSDNAME',
            'CMAUID', 'CMANAME', 'CDTYPE', 'CSDTYPE', 'TYPE', 'pop21', 'LANDAREA'
        ], errors='ignore')  # Use `errors='ignore'` to avoid issues if columns are missing
        # Merge greenspace per capita value with census tract geometry
        greenspace_final_path = os.path.join(output_dir, "greenspace_final.geojson")

        #now we want to just display only the census geometry and per capita data
        # Ensure both GeoDataFrames use the same CRS
        census_path = os.path.join(output_dir, "censustracts.geojson")
        census_gdf = gpd.read_file(census_path)
        census_gdf = census_gdf.to_crs(epsg=4326)
        greenspace_sum = greenspace_sum.to_crs(epsg=4326)

        # Merge on CTUID to get geometry from census_gdf and greenspace_per_capita from greenspace_sum
        GS_capita_gdf = census_gdf[['CTUID', 'geometry']].merge(
            greenspace_sum[['CTUID', 'greenspace_per_capita']],
            on='CTUID',
            how='left'
        )

        # Ensure the GeoDataFrame is in EPSG:4326
        GS_capita_gdf = gpd.GeoDataFrame(GS_capita_gdf, geometry='geometry', crs='EPSG:4326')

        # Save to GeoJSON
        greenspace_final_path = os.path.join(output_dir, "greenspace_per_capita.geojson")
        GS_capita_gdf.to_file(greenspace_final_path, driver="GeoJSON")
        print(f"CRS before saving: {GS_capita_gdf.crs}")
        print(f"Final greenspace data saved to: {greenspace_final_path}")
        
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
# process_greenspace_data("raw-data/montreal/greenspace_capita.geojson", output_dir="data/montreal")


make a full pipline call

In [ ]:
"""
scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF")
#band 4
scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF")
#band 5

calculate_ndvi("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF", "raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF",city="montreal.tiff")
#creates ndvi map
otsu_ndvi_threshold("raw-data/montreal/landsat2_c2/ndvi_montreal.tiff")
# finds optimal threshold value, 
mask_to_geojson("raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff")
# creates mask of greenspace in geojson format
clip_geojson_to_bbox("raw-data/montreal/output_simplified.geojson", output_dir="data/montreal")

#clips to city boundary, will ask for coordinates, in this case, -74.00,45.35,-73.50,45.75
combine_greenspace_with_census("raw-data/montreal/greenspace_only.geojson", output_dir="raw-data/montreal")
# combines greenspace data with census tracts
process_greenspace_census_data("raw-data/montreal/greenspace_with_census.geojson", output_dir="data/montreal")
# processes greenspace and census data to calculate greenspace per capita
process_greenspace_data("raw-data/montreal/greenspace_capita.geojson", output_dir="data/montreal")
# processes and saves final data as per capita
"""





def process_city_pipeline(city_name, base_dir):
    """
    Processes the entire pipeline for a given city.

    Parameters:
        city_name (str): Name of the city (e.g., "montreal").
        base_dir (str): Base directory where the data is stored.

    Outputs:
        Saves all intermediate and final outputs to the appropriate directories.
    """
    import os

    # Define paths
    city_dir = os.path.join(base_dir, "raw-data", city_name)
    output_dir = os.path.join(base_dir, "data", city_name)
    os.makedirs(output_dir, exist_ok=True)

    # File paths
    red_band = os.path.join(city_dir, "landsat2_c2", "LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF")
    nir_band = os.path.join(city_dir, "landsat2_c2", "LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF")
    ndvi_tiff = os.path.join(city_dir, "landsat2_c2", f"ndvi_{city_name}.tiff")
    mask_tiff = os.path.join(city_dir, "landsat2_c2", f"ndvi_{city_name}_mask.tiff")
    
    geojson_mask = os.path.join(city_dir, f"greenspace_ndvi_{city_name}.geojson")
    clipped_geojson = os.path.join(output_dir, "greenspace_only.geojson")
    combined_geojson = os.path.join(city_dir, "greenspace_with_census.geojson")
    capita_geojson = os.path.join(output_dir, "greenspace_capita.geojson")

    # Step 1: Scale TIFF bands
    print(f"Processing city: {city_name}")
    scale_tiff(red_band)
    scale_tiff(nir_band)

    # Step 2: Calculate NDVI
    calculate_ndvi(red_band, nir_band, city=f"{city_name}.tiff")

    # Step 3: Apply Otsu threshold to NDVI
    otsu_ndvi_threshold(ndvi_tiff)

    # Step 4: Convert mask to GeoJSON
    mask_to_geojson(mask_tiff)

    # Step 5: Clip GeoJSON to bounding box
    clip_geojson_to_bbox(geojson_mask, output_dir)

    # Step 6: Combine greenspace with census data
    combine_greenspace_with_census(clipped_geojson, output_dir)

    # Step 7: Process greenspace census data
    process_greenspace_census_data(combined_geojson, output_dir)

    # Step 8: Final processing for greenspace per capita
    process_greenspace_data(capita_geojson, output_dir)

    print(f"Pipeline completed for city: {city_name}")

In [ ]:
def clip_final_geojson_files(bbox_coords, base_dir, city_name):
    """
    Clips the greenspace_per_capita and censustracts GeoJSON files to the specified bounding box.

    Parameters:
        bbox_coords (tuple): Bounding box coordinates as (minx, miny, maxx, maxy).
        base_dir (str): Base directory where the data is stored.
        city_name (str): Name of the city (e.g., "montreal").

    Outputs:
        Saves clipped versions of both GeoJSON files to the data/{city_name} directory.
    """
    import geopandas as gpd
    from shapely.geometry import box
    import os

    os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'
    
    minx, miny, maxx, maxy = bbox_coords
    bbox = box(minx, miny, maxx, maxy)
    
    output_dir = os.path.join(base_dir, "data", city_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # File paths - both files are now in the city-specific directory
    greenspace_per_capita_file = os.path.join(output_dir, "greenspace_per_capita.geojson")
    censustracts_file = os.path.join(output_dir, "censustracts.geojson")
    
    print(f"Clipping files to bounding box: ({minx}, {miny}, {maxx}, {maxy})")
    
    # Clip greenspace_per_capita.geojson
    if os.path.exists(greenspace_per_capita_file):
        try:
            gdf = gpd.read_file(greenspace_per_capita_file)
            gdf = gdf.to_crs(epsg=4326)
            clipped_gdf = gdf.clip(bbox)
            
            # Save clipped file (overwrite original)
            clipped_gdf.to_file(greenspace_per_capita_file, driver="GeoJSON")
            print(f"✓ Clipped greenspace_per_capita.geojson saved to: {greenspace_per_capita_file}")
        except Exception as e:
            print(f"✗ Error clipping greenspace_per_capita.geojson: {e}")
    else:
        print(f"⚠ greenspace_per_capita.geojson not found at: {greenspace_per_capita_file}")
    
    # Clip censustracts.geojson (overwrite original in city directory)
    if os.path.exists(censustracts_file):
        try:
            gdf = gpd.read_file(censustracts_file)
            gdf = gdf.to_crs(epsg=4326)
            clipped_gdf = gdf.clip(bbox)
            
            # Save clipped census tracts (overwrite original)
            clipped_gdf.to_file(censustracts_file, driver="GeoJSON")
            print(f"✓ Clipped censustracts.geojson saved to: {censustracts_file}")
        except Exception as e:
            print(f"✗ Error clipping censustracts.geojson: {e}")
    else:
        print(f"⚠ censustracts.geojson not found at: {censustracts_file}")
    
    print("Clipping completed!")

# Example usage:
# bbox_ottawa = (-76.00, 45.26, -75.29, 45.64)
# clip_final_geojson_files(bbox_ottawa, r"D:\Desktop\greenspace_web\py_data_process", "ottawa")

In [ ]:
"""
scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF")
#band 4
scale_tiff("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF")
#band 5

calculate_ndvi("raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B4.TIF", "raw-data/montreal/landsat2_c2/LC08_L2SP_014028_20240701_20240711_02_T1_SR_B5.TIF",city="montreal.tiff")
#creates ndvi map
otsu_ndvi_threshold("raw-data/montreal/landsat2_c2/ndvi_montreal.tiff")
# finds optimal threshold value, 
mask_to_geojson("raw-data/montreal/landsat2_c2/ndvi_montreal_mask.tiff")
# creates mask of greenspace in geojson format
clip_geojson_to_bbox("raw-data/montreal/output_simplified.geojson", output_dir="data/montreal")

#clips to city boundary, will ask for coordinates, in this case, -74.00,45.35,-73.50,45.75
combine_greenspace_with_census("raw-data/montreal/greenspace_only.geojson", output_dir="raw-data/montreal")
# combines greenspace data with census tracts
process_greenspace_census_data("raw-data/montreal/greenspace_with_census.geojson", output_dir="data/montreal")
# processes greenspace and census data to calculate greenspace per capita
process_greenspace_data("raw-data/montreal/greenspace_capita.geojson", output_dir="data/montreal")
# processes and saves final data as per capita
"""




def process_city_pipeline(city_name, base_dir, census, census_boundaries, bbox_coords):
    """
    Processes the entire pipeline for a given city.

    Parameters:
        city_name (str): Name of the city (e.g., "montreal").
        base_dir (str): Base directory where the data is stored.
        census (str): Path to census CSV file.
        census_boundaries (str): Path to census shapefile.
        bbox_coords (tuple): Bounding box coordinates as (minx, miny, maxx, maxy).

    Outputs:
        Saves all intermediate and final outputs to the appropriate directories.
    """
    import os
    import glob
    os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

    # Define paths
    city_dir = os.path.join(base_dir, "raw-data", city_name)
    output_dir = os.path.join(base_dir, "data", city_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Step 0: Create census geojson for the city
    create_census_geojson(census, census_boundaries, output_dir)

    # File paths
    # Dynamically find the red and NIR band files
    red_band_list = glob.glob(os.path.join(city_dir, "landsat2_c2", "*_B4.TIF"))
    nir_band_list = glob.glob(os.path.join(city_dir, "landsat2_c2", "*_B5.TIF"))

    if not red_band_list or not nir_band_list:
        raise FileNotFoundError("landsat Red or NIR band files not found in the expected directory.")

    # Extract the first match (convert list to string)
    red_band = red_band_list[0]
    nir_band = nir_band_list[0]

    print(f"Found red band: {red_band}")
    print(f"Found NIR band: {nir_band}")
    
    ndvi_tiff = os.path.join(city_dir, "landsat2_c2", f"ndvi_{city_name}.tiff")
    mask_tiff = os.path.join(city_dir, "landsat2_c2", f"ndvi_{city_name}_mask.tiff")
    
    geojson_mask = os.path.join(city_dir, f"greenspace_ndvi_{city_name}.geojson")
    # Use city_dir consistently since clip_geojson_to_bbox saves to both locations
    clipped_geojson = os.path.join(city_dir, "greenspace_only.geojson")
    combined_geojson = os.path.join(city_dir, "greenspace_with_census.geojson")
    capita_geojson = os.path.join(city_dir, "greenspace_capita.geojson")

    # Step 1: Scale TIFF bands
    print(f"Processing city: {city_name}")
    scale_tiff(red_band)
    scale_tiff(nir_band)

    # Step 2: Calculate NDVI
    calculate_ndvi(red_band, nir_band, city=f"{city_name}.tiff")

    # Step 3: Apply Otsu threshold to NDVI
    otsu_ndvi_threshold(ndvi_tiff)

    # Step 4: Convert mask to GeoJSON
    mask_to_geojson(mask_tiff)

    # Step 5: Clip GeoJSON to bounding box
    # Pass output_dir but it also saves to city_dir
    clip_geojson_to_bbox(geojson_mask, output_dir, bbox_coords=bbox_coords)

    # Step 6: Combine greenspace with census data
    # Pass city_dir as output_dir so it saves greenspace_with_census.geojson there
    combine_greenspace_with_census(clipped_geojson, city_dir)

    # Step 7: Process greenspace census data
    # This reads from city_dir and saves capita file there
    process_greenspace_census_data(combined_geojson, output_dir)

    # Step 8: Final processing for greenspace per capita
    # This reads capita file and saves final output to output_dir
    process_greenspace_data(capita_geojson, output_dir)
    
    # Step 9: Clip final GeoJSON files to bounding box
    clip_final_geojson_files(bbox_coords, base_dir, city_name)

    print(f"Pipeline completed for city: {city_name}")


# Example usage:
# bbox_ottawa = (-76.00, 45.26, -75.29, 45.64)
# process_city_pipeline("ottawa", r"D:\Desktop\greenspace_web\py_data_process", 
#                       "raw-data/ottawa/census.csv", "raw-data/census tracts/lct_000b21a_e.shp", 
#                       bbox_ottawa)